In [2]:
# Load and preprocess data
df = pd.read_csv('final_aluminum_data.csv')

# Fill NaN values with 0
df = df.fillna(0)

# Separate features
composition_cols = ['Al', 'Cu', 'Mg', 'Mn', 'Si', 'Zn', 'Fe', 'Ni', 'Cr', 'Ti', 'Pb', 'Sn', 'Zr', 'Co', 'V']
processing_cols = ['solution_temp', 'solution_time', 'aging_temp', 'aging_time', 'strain_hardening_index']
property_cols = ['Tensile Strength (MPa)', 'Yield Strength (MPa)']

# Combine all input features
input_features = composition_cols + processing_cols

# Create custom Dataset class
class AlloyDataset(Dataset):
    def __init__(self, data, input_features):
        self.data = data
        self.features = input_features
        self.scaler = StandardScaler()
        self.X = torch.FloatTensor(self.scaler.fit_transform(data[self.features]))
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.X[idx]
    
    def get_scaler(self):
        return self.scaler

# Create dataset
dataset = AlloyDataset(df, input_features)
print(f"Dataset size: {len(dataset)}")
print(f"Input dimension: {dataset[0].shape[0]}")

# Print summary of the data to verify NaN handling
print("\nVerifying NaN handling:")
print("Number of NaN values in each column:")
print(df[input_features].isna().sum())
print("\nSample of processed data:")
print(df[input_features].head())

Dataset size: 1154
Input dimension: 20

Verifying NaN handling:
Number of NaN values in each column:
Al                        0
Cu                        0
Mg                        0
Mn                        0
Si                        0
Zn                        0
Fe                        0
Ni                        0
Cr                        0
Ti                        0
Pb                        0
Sn                        0
Zr                        0
Co                        0
V                         0
solution_temp             0
solution_time             0
aging_temp                0
aging_time                0
strain_hardening_index    0
dtype: int64

Sample of processed data:
        Al      Cu      Mg   Mn       Si      Zn       Fe   Ni   Cr   Ti   Pb  \
0  0.88011  0.0198  0.0212  0.0  0.00034  0.0768  0.00055  0.0  0.0  0.0  0.0   
1  0.88011  0.0198  0.0212  0.0  0.00034  0.0768  0.00055  0.0  0.0  0.0  0.0   
2  0.99450  0.0000  0.0000  0.0  0.00000  0.0000  0.0000

In [9]:
new_compositions.to_csv('new_compositions.csv', index=False)

# Exploratory Data Analysis of Aluminum Materials Data

This notebook performs a comprehensive exploratory data analysis of the aluminum materials dataset. We'll analyze the data structure, distributions, relationships, and identify any patterns or insights in the data.

## 1. Load and Examine Dataset

First, let's import the required libraries and load our dataset.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# Set basic plotting style
plt.style.use('default')

# Load the dataset
df = pd.read_csv('al_data.csv')

# Display basic information about the dataset
print("Dataset Shape:", df.shape)
print("\nDataset Info:")
df.info()
print("\nFirst few rows of the dataset:")
df.head()

Dataset Shape: (1154, 31)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1154 entries, 0 to 1153
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              1154 non-null   int64  
 1   Processing              1154 non-null   object 
 2   Ag                      1154 non-null   float64
 3   Al                      1154 non-null   float64
 4   B                       1154 non-null   float64
 5   Be                      1154 non-null   float64
 6   Bi                      1154 non-null   float64
 7   Cd                      1154 non-null   float64
 8   Co                      1154 non-null   float64
 9   Cr                      1154 non-null   float64
 10  Cu                      1154 non-null   float64
 11  Er                      1154 non-null   float64
 12  Eu                      1154 non-null   float64
 13  Fe                      1154 non-null   float64
 14 

,Unnamed: 0,Processing,Ag,Al,B,Be,Bi,Cd,Co,Cr,...,Si,Sn,Ti,V,Zn,Zr,Elongation (%),Tensile Strength (MPa),Yield Strength (MPa),class
0,0,Solutionised + Artificially peak aged,0.0,0.88011,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00034,0.0,0.0,0.0,0.0768,0.0012,16.8,651.6,583.3,2
1,1,Solutionised + Artificially over aged,0.0,0.88011,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00034,0.0,0.0,0.0,0.0768,0.0012,15.4,557.0,513.0,4
2,2,Solutionised + Artificially peak aged,0.0,0.99450,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0000,0.0000,10.5,320.0,300.0,2
3,3,Solutionised + Artificially peak aged,0.0,0.99250,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0000,0.0000,4.5,280.0,265.0,2
4,4,Solutionised + Artificially peak aged,0.0,0.99000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0000,0.0000,7.0,325.0,290.0,2


## 2. Data Cleaning and Missing Values Analysis

Let's check for missing values, duplicates, and any data quality issues that need to be addressed.

In [2]:
# Check for missing values
print("Missing values in each column:")
print(df.isnull().sum())

# Check for duplicates
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Get basic statistics of numerical columns
print("\nBasic statistics of numerical columns:")
print(df.describe())

Missing values in each column:
Unnamed: 0                  0
Processing                  0
Ag                          0
Al                          0
B                           0
Be                          0
Bi                          0
Cd                          0
Co                          0
Cr                          0
Cu                          0
Er                          0
Eu                          0
Fe                          0
Ga                          0
Li                          0
Mg                          0
Mn                          0
Ni                          0
Pb                          0
Sc                          0
Si                          0
Sn                          0
Ti                          0
V                           0
Zn                          0
Zr                          0
Elongation (%)            108
Tensile Strength (MPa)     33
Yield Strength (MPa)      109
class                       0
dtype: int64

Number of duplicate rows:

## 3. Distribution Analysis

Let's analyze the distribution of numerical variables in our dataset.

In [3]:
# Select numerical columns
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Create histograms for numerical variables
fig = make_subplots(rows=len(numerical_cols), cols=1,
                    subplot_titles=numerical_cols)

for i, col in enumerate(numerical_cols, 1):
    fig.add_trace(
        go.Histogram(x=df[col], name=col, nbinsx=30),
        row=i, col=1
    )

fig.update_layout(height=300*len(numerical_cols), 
                 title_text="Distribution of Numerical Variables",
                 showlegend=False)
fig.show()

# Add basic statistics and normality tests
for col in numerical_cols:
    print(f"\nVariable: {col}")
    print("Skewness:", stats.skew(df[col].dropna()))
    print("Kurtosis:", stats.kurtosis(df[col].dropna()))
    stat, p_value = stats.normaltest(df[col].dropna())
    print("Normality test p-value:", p_value)


Variable: Unnamed: 0
Skewness: 0.0
Kurtosis: -1.200001802187405
Normality test p-value: 2.5280805680124636e-189

Variable: Ag
Skewness: 7.96281935197556
Kurtosis: 66.84394094427302
Normality test p-value: 0.0

Variable: Al
Skewness: -1.162752636043663
Kurtosis: 2.368635965167149
Normality test p-value: 3.6023526135648686e-53

Variable: B
Skewness: 7.209085643517907
Kurtosis: 49.97091581557601
Normality test p-value: 0.0

Variable: Be
Skewness: 23.905958333208467
Kurtosis: 570.3076187045419
Normality test p-value: 0.0

Variable: Bi
Skewness: 12.873604052153864
Kurtosis: 164.83520610514526
Normality test p-value: 0.0

Variable: Cd
Skewness: 16.89684817535122
Kurtosis: 283.5034782608697
Normality test p-value: 0.0

Variable: Co
Skewness: 13.334268435573463
Kurtosis: 186.6126987409569
Normality test p-value: 0.0

Variable: Cr
Skewness: 16.627607119987918
Kurtosis: 347.7315776412586
Normality test p-value: 0.0

Variable: Cu
Skewness: 1.398314676605395
Kurtosis: 0.8650318185845571
Normality

## 4. Correlation Analysis

Let's examine the relationships between numerical variables using a correlation matrix and heatmap.

In [4]:
# Calculate correlation matrix
correlation_matrix = df[numerical_cols].corr()

# Create correlation heatmap using plotly
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix,
    x=numerical_cols,
    y=numerical_cols,
    colorscale='RdBu',
    zmid=0,
))

fig.update_layout(
    title='Correlation Heatmap',
    height=600,
    width=800,
)

fig.show()

# Print strong correlations (absolute value > 0.5)
print("\nStrong correlations (|correlation| > 0.5):")
for i in range(len(numerical_cols)):
    for j in range(i+1, len(numerical_cols)):
        corr = correlation_matrix.iloc[i, j]
        if abs(corr) > 0.5:
            print(f"{numerical_cols[i]} vs {numerical_cols[j]}: {corr:.3f}")


Strong correlations (|correlation| > 0.5):
Unnamed: 0 vs Tensile Strength (MPa): 0.607
Unnamed: 0 vs Yield Strength (MPa): 0.623
Al vs Zn: -0.624
Al vs Tensile Strength (MPa): -0.554
Al vs Yield Strength (MPa): -0.549
B vs Er: 0.612
Be vs Ga: 0.808
Bi vs Pb: 1.000
Zn vs Tensile Strength (MPa): 0.618
Zn vs Yield Strength (MPa): 0.637
Tensile Strength (MPa) vs Yield Strength (MPa): 0.959


## 5. Feature Relationships

Let's visualize the relationships between key features using scatter plots, particularly those with strong correlations.

In [5]:
# Create scatter plots for important relationships
key_features = ['Al', 'Zn', 'Tensile Strength (MPa)', 'Yield Strength (MPa)']

fig = make_subplots(rows=2, cols=2, subplot_titles=[
    'Al vs Tensile Strength',
    'Zn vs Tensile Strength',
    'Al vs Yield Strength',
    'Tensile vs Yield Strength'
])

# Al vs Tensile Strength
fig.add_trace(
    go.Scatter(x=df['Al'], y=df['Tensile Strength (MPa)'], mode='markers', name='Al vs Tensile'),
    row=1, col=1
)

# Zn vs Tensile Strength
fig.add_trace(
    go.Scatter(x=df['Zn'], y=df['Tensile Strength (MPa)'], mode='markers', name='Zn vs Tensile'),
    row=1, col=2
)

# Al vs Yield Strength
fig.add_trace(
    go.Scatter(x=df['Al'], y=df['Yield Strength (MPa)'], mode='markers', name='Al vs Yield'),
    row=2, col=1
)

# Tensile vs Yield Strength
fig.add_trace(
    go.Scatter(x=df['Tensile Strength (MPa)'], y=df['Yield Strength (MPa)'], mode='markers', name='Tensile vs Yield'),
    row=2, col=2
)

fig.update_layout(height=800, width=1000, title_text="Key Feature Relationships", showlegend=True)
fig.show()

# Print summary statistics for key features
print("\nSummary statistics for key features:")
print(df[key_features].describe())


Summary statistics for key features:
                Al           Zn  Tensile Strength (MPa)  Yield Strength (MPa)
count  1154.000000  1154.000000             1121.000000           1045.000000
mean      0.943074     0.012376              344.481985            281.872023
std       0.041144     0.025283              150.924730            149.694416
min       0.749500     0.000000               44.815920             10.342135
25%       0.922300     0.000000              220.632224            163.200000
50%       0.946947     0.000130              324.053579            260.000000
75%       0.974858     0.002500              468.500000            393.001149
max       0.999900     0.120000              820.000000            790.000000


## 6. Data Preprocessing and Feature Engineering

Let's preprocess our data by:
1. Identifying and separating features and target variables
2. Normalizing numerical features
3. Handling any categorical variables
4. Creating processed dataset for modeling

In [2]:
# Import preprocessing tools
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify feature types
numerical_features = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
target_variables = ['Tensile Strength (MPa)', 'Yield Strength (MPa)']
composition_features = [col for col in numerical_features if col not in target_variables and col != 'Unnamed: 0']

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('composition_scaler', RobustScaler(), composition_features),
        ('target_scaler', StandardScaler(), target_variables)
    ])

# Fit and transform the data
processed_data = preprocessor.fit_transform(df[composition_features + target_variables])
processed_df = pd.DataFrame(
    processed_data, 
    columns=composition_features + target_variables
)

# Display first few rows of processed data
print("First few rows of processed data:")
print(processed_df.head())

# Compare distributions before and after preprocessing
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    'Original Al Distribution',
    'Processed Al Distribution',
    'Original Tensile Strength Distribution',
    'Processed Tensile Strength Distribution'
])

# Original distributions
fig.add_trace(
    go.Histogram(x=df['Al'], name='Original Al'),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=df['Tensile Strength (MPa)'], name='Original Tensile'),
    row=2, col=1
)

# Processed distributions
fig.add_trace(
    go.Histogram(x=processed_df['Al'], name='Processed Al'),
    row=1, col=2
)
fig.add_trace(
    go.Histogram(x=processed_df['Tensile Strength (MPa)'], name='Processed Tensile'),
    row=2, col=2
)

fig.update_layout(height=800, width=1000, title_text="Original vs Processed Distributions")
fig.show()

# Print summary statistics of processed data
print("\nSummary statistics of processed data:")
print(processed_df.describe())

First few rows of processed data:
    Ag        Al    B   Be   Bi   Cd   Co   Cr        Cu   Er  ...      Sc  \
0  0.0 -1.271686  0.0  0.0  0.0  0.0  0.0  0.0  0.864078  0.0  ...  0.0000   
1  0.0 -1.271686  0.0  0.0  0.0  0.0  0.0  0.0  0.864078  0.0  ...  0.0000   
2  0.0  0.904777  0.0  0.0  0.0  0.0  0.0  0.0 -0.097087  0.0  ...  0.0055   
3  0.0  0.866724  0.0  0.0  0.0  0.0  0.0  0.0 -0.097087  0.0  ...  0.0075   
4  0.0  0.819157  0.0  0.0  0.0  0.0  0.0  0.0 -0.097087  0.0  ...  0.0100   

      Si   Sn   Ti    V      Zn      Zr  Elongation (%)  \
0 -0.132  0.0  0.0  0.0  30.668  0.0012        0.694611   
1 -0.132  0.0  0.0  0.0  30.668  0.0012        0.526946   
2 -0.200  0.0  0.0  0.0  -0.052  0.0000       -0.059880   
3 -0.200  0.0  0.0  0.0  -0.052  0.0000       -0.778443   
4 -0.200  0.0  0.0  0.0  -0.052  0.0000       -0.479042   

   Tensile Strength (MPa)  Yield Strength (MPa)  
0                2.035817              2.014586  
1                1.408734              1.5


Summary statistics of processed data:
                Ag           Al            B            Be          Bi  \
count  1154.000000  1154.000000  1154.000000  1.154000e+03  1154.00000   
mean      0.000078    -0.073694     0.000002  8.318891e-08     0.00003   
std       0.000553     0.782837     0.000013  1.873727e-06     0.00039   
min       0.000000    -3.756761     0.000000  0.000000e+00     0.00000   
25%       0.000000    -0.468950     0.000000  0.000000e+00     0.00000   
50%       0.000000     0.000000     0.000000  0.000000e+00     0.00000   
75%       0.000000     0.531050     0.000000  0.000000e+00     0.00000   
max       0.007000     1.007521     0.000100  4.500000e-05     0.00549   

                Cd           Co           Cr           Cu           Er  ...  \
count  1154.000000  1154.000000  1154.000000  1154.000000  1154.000000  ...   
mean      0.000007     0.000045     0.594860     0.538400     0.000064  ...   
std       0.000118     0.000528     1.587609     0.898034

In [3]:
# Analyze the impact of preprocessing
print("Impact of preprocessing on feature scales:")
print("\nOriginal data ranges:")
for col in composition_features:
    print(f"{col}: [{df[col].min():.3f}, {df[col].max():.3f}]")
    
print("\nProcessed data ranges:")
for col in composition_features:
    print(f"{col}: [{processed_df[col].min():.3f}, {processed_df[col].max():.3f}]")

# Calculate correlation matrix for processed data
processed_corr = processed_df.corr()

# Create correlation heatmap for processed data
fig = go.Figure(data=go.Heatmap(
    z=processed_corr,
    x=processed_df.columns,
    y=processed_df.columns,
    colorscale='RdBu',
    zmid=0,
))

fig.update_layout(
    title='Correlation Heatmap (Processed Data)',
    height=800,
    width=1000,
)

fig.show()

Impact of preprocessing on feature scales:

Original data ranges:
Ag: [0.000, 0.007]
Al: [0.750, 1.000]
B: [0.000, 0.000]
Be: [0.000, 0.000]
Bi: [0.000, 0.005]
Cd: [0.000, 0.002]
Co: [0.000, 0.008]
Cr: [0.000, 0.041]
Cu: [0.000, 0.069]
Er: [0.000, 0.004]
Eu: [0.000, 0.001]
Fe: [0.000, 0.012]
Ga: [0.000, 0.000]
Li: [0.000, 0.038]
Mg: [0.000, 0.060]
Mn: [0.000, 0.013]
Ni: [0.000, 0.020]
Pb: [0.000, 0.005]
Sc: [0.000, 0.015]
Si: [0.000, 0.220]
Sn: [0.000, 0.200]
Ti: [0.000, 0.003]
V: [0.000, 0.003]
Zn: [0.000, 0.120]
Zr: [0.000, 0.015]
Elongation (%): [0.500, 50.000]

Processed data ranges:
Ag: [0.000, 0.007]
Al: [-3.757, 1.008]
B: [0.000, 0.000]
Be: [0.000, 0.000]
Bi: [0.000, 0.005]
Cd: [0.000, 0.002]
Co: [0.000, 0.008]
Cr: [0.000, 34.250]
Cu: [-0.097, 3.228]
Er: [0.000, 0.004]
Eu: [0.000, 0.001]
Fe: [-0.300, 4.500]
Ga: [0.000, 0.000]
Li: [0.000, 0.038]
Mg: [-0.488, 2.439]
Mn: [-0.222, 2.689]
Ni: [0.000, 0.020]
Pb: [0.000, 0.005]
Sc: [0.000, 0.015]
Si: [-0.200, 43.800]
Sn: [0.000, 0.200]

## 7. Advanced Data Processing Techniques

Let's explore different processing techniques suitable for materials data:

1. **Scaling Techniques**:
   - Standard Scaling (Z-score normalization)
   - Robust Scaling (handles outliers better)
   - Min-Max Scaling (for bounded features)
   
2. **Feature Engineering**:
   - Ratio features
   - Polynomial features
   - Composition ratios
   
3. **Dimension Reduction**:
   - PCA (Principal Component Analysis)
   - Feature selection based on importance

In [5]:
# Compare different scaling techniques
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import pandas as pd
import numpy as np

# Select features for comparison
features_to_scale = ['Al', 'Cu', 'Mg', 'Zn']
scaling_data = df[features_to_scale].copy()

# Apply different scaling techniques
scalers = {
    'Standard': StandardScaler(),
    'MinMax': MinMaxScaler(),
    'Robust': RobustScaler()
}

scaled_dfs = {}
for name, scaler in scalers.items():
    scaled_data = scaler.fit_transform(scaling_data)
    scaled_dfs[name] = pd.DataFrame(scaled_data, columns=features_to_scale)

# Compare distributions
fig = make_subplots(rows=len(features_to_scale), cols=4,
                    subplot_titles=['Original'] + list(scalers.keys()) * len(features_to_scale))

for i, feature in enumerate(features_to_scale, 1):
    # Original distribution
    fig.add_trace(
        go.Histogram(x=scaling_data[feature], name=f'Original {feature}'),
        row=i, col=1
    )
    
    # Scaled distributions
    for j, (name, scaled_df) in enumerate(scaled_dfs.items(), 2):
        fig.add_trace(
            go.Histogram(x=scaled_df[feature], name=f'{name} {feature}'),
            row=i, col=j
        )

fig.update_layout(height=1000, width=1200, 
                 title_text="Comparison of Scaling Techniques",
                 showlegend=False)
fig.show()

# Print statistics for each scaling method
for feature in features_to_scale:
    print(f"\nFeature: {feature}")
    print("Original Stats:")
    print(f"Mean: {scaling_data[feature].mean():.3f}")
    print(f"Std: {scaling_data[feature].std():.3f}")
    print(f"Range: [{scaling_data[feature].min():.3f}, {scaling_data[feature].max():.3f}]")
    
    for name, scaled_df in scaled_dfs.items():
        print(f"\n{name} Scaling Stats:")
        print(f"Mean: {scaled_df[feature].mean():.3f}")
        print(f"Std: {scaled_df[feature].std():.3f}")
        print(f"Range: [{scaled_df[feature].min():.3f}, {scaled_df[feature].max():.3f}]")


Feature: Al
Original Stats:
Mean: 0.943
Std: 0.041
Range: [0.750, 1.000]

Standard Scaling Stats:
Mean: 0.000
Std: 1.000
Range: [-4.707, 1.382]

MinMax Scaling Stats:
Mean: 0.773
Std: 0.164
Range: [0.000, 1.000]

Robust Scaling Stats:
Mean: -0.074
Std: 0.783
Range: [-3.757, 1.008]

Feature: Cu
Original Stats:
Mean: 0.013
Std: 0.018
Range: [0.000, 0.069]

Standard Scaling Stats:
Mean: 0.000
Std: 1.000
Range: [-0.708, 2.996]

MinMax Scaling Stats:
Mean: 0.191
Std: 0.270
Range: [0.000, 1.000]

Robust Scaling Stats:
Mean: 0.538
Std: 0.898
Range: [-0.097, 3.228]

Feature: Mg
Original Stats:
Mean: 0.015
Std: 0.013
Range: [0.000, 0.060]

Standard Scaling Stats:
Mean: 0.000
Std: 1.000
Range: [-1.103, 3.400]

MinMax Scaling Stats:
Mean: 0.245
Std: 0.222
Range: [0.000, 1.000]

Robust Scaling Stats:
Mean: 0.229
Std: 0.650
Range: [-0.488, 2.439]

Feature: Zn
Original Stats:
Mean: 0.012
Std: 0.025
Range: [0.000, 0.120]

Standard Scaling Stats:
Mean: -0.000
Std: 1.000
Range: [-0.490, 4.259]

MinMax

In [6]:
# Feature Engineering for Materials Data
from sklearn.preprocessing import PolynomialFeatures

# 1. Create ratio features for main alloying elements
ratio_features = pd.DataFrame()
main_elements = ['Al', 'Cu', 'Mg', 'Zn']

# Calculate ratios between main elements
for i, elem1 in enumerate(main_elements):
    for elem2 in main_elements[i+1:]:
        ratio_name = f'{elem1}_{elem2}_ratio'
        ratio_features[ratio_name] = df[elem1] / (df[elem2] + 1e-6)  # Add small constant to avoid division by zero

# 2. Create polynomial features for main elements
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df[main_elements])
poly_feature_names = [f"{main_elements[i]}_{main_elements[j]}" 
                     for i in range(len(main_elements)) 
                     for j in range(i, len(main_elements))]
poly_features_df = pd.DataFrame(poly_features[:, len(main_elements):], 
                              columns=poly_feature_names)

# 3. Calculate total alloying content
ratio_features['total_alloying'] = df[main_elements].sum(axis=1)

# Display the engineered features
print("Sample of ratio features:")
print(ratio_features.head())
print("\nSample of polynomial interaction features:")
print(poly_features_df.head())

# Visualize relationship between engineered features and target
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    'Al/Cu Ratio vs Tensile Strength',
    'Total Alloying vs Tensile Strength',
    'Al*Cu Interaction vs Tensile Strength',
    'Al*Zn Interaction vs Tensile Strength'
])

# Plot ratio feature
fig.add_trace(
    go.Scatter(x=ratio_features['Al_Cu_ratio'], 
               y=df['Tensile Strength (MPa)'], 
               mode='markers', 
               name='Al/Cu Ratio'),
    row=1, col=1
)

# Plot total alloying
fig.add_trace(
    go.Scatter(x=ratio_features['total_alloying'], 
               y=df['Tensile Strength (MPa)'], 
               mode='markers', 
               name='Total Alloying'),
    row=1, col=2
)

# Plot polynomial features
fig.add_trace(
    go.Scatter(x=poly_features_df['Al_Cu'], 
               y=df['Tensile Strength (MPa)'], 
               mode='markers', 
               name='Al*Cu'),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=poly_features_df['Al_Zn'], 
               y=df['Tensile Strength (MPa)'], 
               mode='markers', 
               name='Al*Zn'),
    row=2, col=2
)

fig.update_layout(height=800, width=1000, 
                 title_text="Engineered Features vs Tensile Strength",
                 showlegend=True)
fig.show()

Sample of ratio features:
     Al_Cu_ratio    Al_Mg_ratio    Al_Zn_ratio  Cu_Mg_ratio  Cu_Zn_ratio  \
0      44.447755      41.512664      11.459616     0.933918     0.257809   
1      44.447755      41.512664      11.459616     0.933918     0.257809   
2  994500.000000  994500.000000  994500.000000     0.000000     0.000000   
3  992500.000000  992500.000000  992500.000000     0.000000     0.000000   
4  990000.000000  990000.000000  990000.000000     0.000000     0.000000   

   Mg_Zn_ratio  total_alloying  
0     0.276038         0.99791  
1     0.276038         0.99791  
2     0.000000         0.99450  
3     0.000000         0.99250  
4     0.000000         0.99000  

Sample of polynomial interaction features:
      Al_Al     Al_Cu     Al_Mg     Al_Zn     Cu_Cu    Cu_Mg     Cu_Zn  \
0  0.774594  0.017426  0.018658  0.067592  0.000392  0.00042  0.001521   
1  0.774594  0.017426  0.018658  0.067592  0.000392  0.00042  0.001521   
2  0.989030  0.000000  0.000000  0.000000  0.000000  

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1154 entries, 0 to 1153
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              1154 non-null   int64  
 1   Processing              1154 non-null   object 
 2   Ag                      1154 non-null   float64
 3   Al                      1154 non-null   float64
 4   B                       1154 non-null   float64
 5   Be                      1154 non-null   float64
 6   Bi                      1154 non-null   float64
 7   Cd                      1154 non-null   float64
 8   Co                      1154 non-null   float64
 9   Cr                      1154 non-null   float64
 10  Cu                      1154 non-null   float64
 11  Er                      1154 non-null   float64
 12  Eu                      1154 non-null   float64
 13  Fe                      1154 non-null   float64
 14  Ga                      1154 non-null   

## 8. Data Type Conversion and Label Encoding

Let's ensure all our numerical data is properly converted to float64 format and handle any categorical variables if present.

In [7]:
# Import necessary libraries
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Create a copy of the original dataframe
df_processed = df.copy()

# Display original data types
print("Original data types:")
print(df_processed.dtypes)

# Convert all numeric columns to float64
numeric_columns = df_processed.select_dtypes(include=['float64', 'float32', 'int64', 'int32']).columns
for col in numeric_columns:
    df_processed[col] = df_processed[col].astype('float64')

# Display new data types
print("\nConverted data types:")
print(df_processed.dtypes)

# Compare memory usage before and after conversion
print("\nMemory usage comparison:")
print("Original dataframe memory usage:", df.memory_usage().sum() / 1024, "KB")
print("Processed dataframe memory usage:", df_processed.memory_usage().sum() / 1024, "KB")

# Verify numerical precision
print("\nNumerical precision comparison (first 5 rows of key features):")
key_features = ['Al', 'Cu', 'Mg', 'Zn', 'Tensile Strength (MPa)', 'Yield Strength (MPa)']
print("\nOriginal values:")
print(df[key_features].head())
print("\nProcessed values (float64):")
print(df_processed[key_features].head())

Original data types:
Unnamed: 0                  int64
Processing                 object
Ag                        float64
Al                        float64
B                         float64
Be                        float64
Bi                        float64
Cd                        float64
Co                        float64
Cr                        float64
Cu                        float64
Er                        float64
Eu                        float64
Fe                        float64
Ga                        float64
Li                        float64
Mg                        float64
Mn                        float64
Ni                        float64
Pb                        float64
Sc                        float64
Si                        float64
Sn                        float64
Ti                        float64
V                         float64
Zn                        float64
Zr                        float64
Elongation (%)            float64
Tensile Strength (MPa)    f

In [8]:
# Create normalized version of the dataset with consistent float64 type
# Apply RobustScaler to the processed data
scaler = RobustScaler()
scaled_features = scaler.fit_transform(df_processed[key_features])
df_scaled = pd.DataFrame(scaled_features, columns=key_features)

# Verify data types of scaled data
print("\nScaled data types:")
print(df_scaled.dtypes)

# Compare statistics before and after scaling
print("\nOriginal data statistics:")
print(df_processed[key_features].describe())
print("\nScaled data statistics:")
print(df_scaled.describe())

# Visualize the distributions of original vs scaled data
fig = make_subplots(rows=3, cols=2, subplot_titles=key_features)

for idx, feature in enumerate(key_features):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    # Original distribution
    fig.add_trace(
        go.Histogram(x=df_processed[feature], name=f'Original {feature}', 
                    opacity=0.7, nbinsx=30),
        row=row, col=col
    )
    
    # Scaled distribution
    fig.add_trace(
        go.Histogram(x=df_scaled[feature], name=f'Scaled {feature}',
                    opacity=0.7, nbinsx=30),
        row=row, col=col
    )

fig.update_layout(height=1000, width=1000, 
                 title_text="Original vs Scaled Distributions (float64)",
                 showlegend=True)
fig.show()

# Save the processed and scaled datasets
df_processed.to_csv('processed_aluminum_data.csv', index=False)
df_scaled.to_csv('scaled_aluminum_data.csv', index=False)

print("\nProcessed datasets have been saved as 'processed_aluminum_data.csv' and 'scaled_aluminum_data.csv'")


Scaled data types:
Al                        float64
Cu                        float64
Mg                        float64
Zn                        float64
Tensile Strength (MPa)    float64
Yield Strength (MPa)      float64
dtype: object

Original data statistics:
                Al           Cu           Mg           Zn  \
count  1154.000000  1154.000000  1154.000000  1154.000000   
mean      0.943074     0.013091     0.014696     0.012376   
std       0.041144     0.018500     0.013329     0.025283   
min       0.749500     0.000000     0.000000     0.000000   
25%       0.922300     0.000000     0.004500     0.000000   
50%       0.946947     0.002000     0.010000     0.000130   
75%       0.974858     0.020600     0.025000     0.002500   
max       0.999900     0.068500     0.060000     0.120000   

       Tensile Strength (MPa)  Yield Strength (MPa)  
count             1121.000000           1045.000000  
mean               344.481985            281.872023  
std                150.


Processed datasets have been saved as 'processed_aluminum_data.csv' and 'scaled_aluminum_data.csv'


In [9]:
df

,Unnamed: 0,Processing,Ag,Al,B,Be,Bi,Cd,Co,Cr,...,Si,Sn,Ti,V,Zn,Zr,Elongation (%),Tensile Strength (MPa),Yield Strength (MPa),class
0,0,Solutionised + Artificially peak aged,0.0,0.880110,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.00034,0.0,0.000000,0.0,0.076800,0.0012,16.8,651.6,583.3,2
1,1,Solutionised + Artificially over aged,0.0,0.880110,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.00034,0.0,0.000000,0.0,0.076800,0.0012,15.4,557.0,513.0,4
2,2,Solutionised + Artificially peak aged,0.0,0.994500,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.00000,0.0,0.000000,0.0,0.000000,0.0000,10.5,320.0,300.0,2
3,3,Solutionised + Artificially peak aged,0.0,0.992500,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.00000,0.0,0.000000,0.0,0.000000,0.0000,4.5,280.0,265.0,2
4,4,Solutionised + Artificially peak aged,0.0,0.990000,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.00000,0.0,0.000000,0.0,0.000000,0.0000,7.0,325.0,290.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1149,1149,No Processing,0.0,0.992713,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.00004,0.0,0.000016,0.0,0.000006,0.0000,32.0,110.0,105.0,1
1150,1150,No Processing,0.0,0.967475,0.0,0.0,0.0,0.0,0.0,0.000020,...,0.00087,0.0,0.000300,0.0,0.000025,0.0000,14.2,195.0,58.0,1
1151,1151,No Processing,0.0,0.966676,0.0,0.0,0.0,0.0,0.0,0.000014,...,0.00110,0.0,0.000230,0.0,0.000010,0.0014,10.7,210.0,110.0,1
1152,1152,Strain Harderned (Hard),0.0,0.955000,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.00000,0.0,0.000000,0.0,0.000000,0.0000,NaN,460.0,414.0,5


## 9. Unique Values Analysis

Let's examine the unique values in both original and processed datasets to understand the data distribution better.

In [10]:
# Analyze unique values in original and processed datasets
def analyze_unique_values(dataframe, columns):
    unique_values = {}
    for col in columns:
        n_unique = dataframe[col].nunique()
        unique_vals = dataframe[col].value_counts().head()
        unique_values[col] = {
            'count': n_unique,
            'top_values': unique_vals
        }
    return unique_values

# Analyze original data
print("Unique values in original dataset:")
print("-" * 50)
original_uniques = analyze_unique_values(df, key_features)
for col, info in original_uniques.items():
    print(f"\n{col}:")
    print(f"Number of unique values: {info['count']}")
    print("Top 5 most common values:")
    print(info['top_values'])

# Analyze processed data
print("\nUnique values in processed dataset (after RobustScaler):")
print("-" * 50)
processed_uniques = analyze_unique_values(df_scaled, key_features)
for col, info in processed_uniques.items():
    print(f"\n{col}:")
    print(f"Number of unique values: {info['count']}")
    print("Top 5 most common values:")
    print(info['top_values'])

# Visualize value distributions
fig = make_subplots(rows=2, cols=3, subplot_titles=key_features)

for idx, feature in enumerate(key_features):
    row = idx // 3 + 1
    col = idx % 3 + 1
    
    # Create box plot
    fig.add_trace(
        go.Box(y=df[feature], name='Original', boxpoints='all', jitter=0.3,
               pointpos=-1.8, boxmean=True),
        row=row, col=col
    )
    
    fig.add_trace(
        go.Box(y=df_scaled[feature], name='Processed', boxpoints='all', jitter=0.3,
               pointpos=1.8, boxmean=True),
        row=row, col=col
    )

fig.update_layout(height=800, width=1200, 
                 title_text="Distribution of Values: Original vs Processed",
                 showlegend=True)
fig.show()

# Calculate summary statistics
print("\nSummary of value ranges:")
print("-" * 50)
for feature in key_features:
    print(f"\n{feature}:")
    print("Original Data:")
    print(f"Range: [{df[feature].min():.3f}, {df[feature].max():.3f}]")
    print(f"Unique values: {df[feature].nunique()}")
    print("Zero values: ", (df[feature] == 0).sum())
    
    print("\nProcessed Data:")
    print(f"Range: [{df_scaled[feature].min():.3f}, {df_scaled[feature].max():.3f}]")
    print(f"Unique values: {df_scaled[feature].nunique()}")
    print("Zero values: ", (df_scaled[feature] == 0).sum())

Unique values in original dataset:
--------------------------------------------------

Al:
Number of unique values: 345
Top 5 most common values:
Al
0.9890    35
0.9920    25
0.9340    23
0.9350    20
0.9625    17
Name: count, dtype: int64

Cu:
Number of unique values: 162
Top 5 most common values:
Cu
0.0000    361
0.0010     42
0.0025     30
0.0440     26
0.0450     23
Name: count, dtype: int64

Mg:
Number of unique values: 176
Top 5 most common values:
Mg
0.000    142
0.010     50
0.015     45
0.025     38
0.007     32
Name: count, dtype: int64

Zn:
Number of unique values: 123
Top 5 most common values:
Zn
0.000000    554
0.002500     49
0.001000     40
0.001251     36
0.001251     22
Name: count, dtype: int64

Tensile Strength (MPa):
Number of unique values: 463
Top 5 most common values:
Tensile Strength (MPa)
240.0    18
180.0    13
310.0    13
400.0    13
485.0    11
Name: count, dtype: int64

Yield Strength (MPa):
Number of unique values: 434
Top 5 most common values:
Yield Stren


Summary of value ranges:
--------------------------------------------------

Al:
Original Data:
Range: [0.750, 1.000]
Unique values: 345
Zero values:  0

Processed Data:
Range: [-3.757, 1.008]
Unique values: 345
Zero values:  7

Cu:
Original Data:
Range: [0.000, 0.069]
Unique values: 162
Zero values:  361

Processed Data:
Range: [-0.097, 3.228]
Unique values: 162
Zero values:  8

Mg:
Original Data:
Range: [0.000, 0.060]
Unique values: 176
Zero values:  142

Processed Data:
Range: [-0.488, 2.439]
Unique values: 176
Zero values:  50

Zn:
Original Data:
Range: [0.000, 0.120]
Unique values: 123
Zero values:  554

Processed Data:
Range: [-0.052, 47.948]
Unique values: 123
Zero values:  6

Tensile Strength (MPa):
Original Data:
Range: [44.816, 820.000]
Unique values: 463
Zero values:  0

Processed Data:
Range: [-1.127, 2.001]
Unique values: 463
Zero values:  3

Yield Strength (MPa):
Original Data:
Range: [10.342, 790.000]
Unique values: 434
Zero values:  0

Processed Data:
Range: [-1.086, 2

In [11]:
# Check all columns in the dataset
print("All columns in the dataset:")
print(df.columns.tolist())

# Display first few rows of the entire dataset to see all columns
print("\nFirst few rows of the complete dataset:")
print(df.head())

All columns in the dataset:
['Unnamed: 0', 'Processing', 'Ag', 'Al', 'B', 'Be', 'Bi', 'Cd', 'Co', 'Cr', 'Cu', 'Er', 'Eu', 'Fe', 'Ga', 'Li', 'Mg', 'Mn', 'Ni', 'Pb', 'Sc', 'Si', 'Sn', 'Ti', 'V', 'Zn', 'Zr', 'Elongation (%)', 'Tensile Strength (MPa)', 'Yield Strength (MPa)', 'class']

First few rows of the complete dataset:
   Unnamed: 0                              Processing   Ag       Al    B   Be  \
0           0  Solutionised  + Artificially peak aged  0.0  0.88011  0.0  0.0   
1           1   Solutionised + Artificially over aged  0.0  0.88011  0.0  0.0   
2           2  Solutionised  + Artificially peak aged  0.0  0.99450  0.0  0.0   
3           3  Solutionised  + Artificially peak aged  0.0  0.99250  0.0  0.0   
4           4  Solutionised  + Artificially peak aged  0.0  0.99000  0.0  0.0   

    Bi   Cd   Co   Cr  ...       Si   Sn   Ti    V      Zn      Zr  \
0  0.0  0.0  0.0  0.0  ...  0.00034  0.0  0.0  0.0  0.0768  0.0012   
1  0.0  0.0  0.0  0.0  ...  0.00034  0.0  0.0  0.0

## 10. Processing Conditions Analysis

Let's analyze the unique processing conditions in the dataset and their distribution.

In [12]:
# Let's read the data again to make sure we have all columns
df_complete = pd.read_csv('al_data.csv')

# Check if Processing column exists
if 'Processing' in df_complete.columns:
    # Get unique processing conditions
    processing_values = df_complete['Processing'].unique()
    processing_counts = df_complete['Processing'].value_counts()
    
    print("Unique Processing Conditions:")
    print("-" * 50)
    print("\nNumber of unique processing conditions:", len(processing_values))
    print("\nProcessing conditions and their frequencies:")
    print(processing_counts)
    
    # Create a bar plot of processing conditions
    fig = go.Figure(data=[
        go.Bar(x=processing_counts.index, 
               y=processing_counts.values)
    ])
    
    fig.update_layout(
        title='Distribution of Processing Conditions',
        xaxis_title='Processing Condition',
        yaxis_title='Count',
        height=600,
        width=1000
    )
    
    fig.show()
    
    # Analyze relationship between processing and mechanical properties
    if all(col in df_complete.columns for col in ['Tensile Strength (MPa)', 'Yield Strength (MPa)']):
        fig = make_subplots(rows=1, cols=2,
                           subplot_titles=['Processing vs Tensile Strength',
                                         'Processing vs Yield Strength'])
        
        # Box plot for Tensile Strength
        fig.add_trace(
            go.Box(x=df_complete['Processing'],
                  y=df_complete['Tensile Strength (MPa)'],
                  name='Tensile Strength'),
            row=1, col=1
        )
        
        # Box plot for Yield Strength
        fig.add_trace(
            go.Box(x=df_complete['Processing'],
                  y=df_complete['Yield Strength (MPa)'],
                  name='Yield Strength'),
            row=1, col=2
        )
        
        fig.update_layout(height=600, width=1200,
                         title_text="Mechanical Properties by Processing Condition")
        fig.show()
        
        # Statistical summary by processing condition
        print("\nStatistical Summary by Processing Condition:")
        print("-" * 50)
        print("\nTensile Strength (MPa):")
        print(df_complete.groupby('Processing')['Tensile Strength (MPa)'].describe())
        print("\nYield Strength (MPa):")
        print(df_complete.groupby('Processing')['Yield Strength (MPa)'].describe())
    
else:
    print("Note: 'Processing' column not found in the dataset.")
    print("\nAvailable columns in the dataset:")
    print(df_complete.columns.tolist())

Unique Processing Conditions:
--------------------------------------------------

Number of unique processing conditions: 10

Processing conditions and their frequencies:
Processing
Solutionised  + Artificially peak aged         375
Solutionised + Artificially over aged          214
Strain Harderned (Hard)                        197
No Processing                                  141
Solutionised + Naturally aged                   86
Solutionised + Cold Worked + Naturally aged     57
Strain hardened                                 41
Artificial aged                                 21
Naturally aged                                  18
Solutionised                                     4
Name: count, dtype: int64



Statistical Summary by Processing Condition:
--------------------------------------------------

Tensile Strength (MPa):
                                             count        mean         std  \
Processing                                                                   
Artificial aged                               17.0  222.253934   75.520129   
Naturally aged                                18.0  236.520517   94.746342   
No Processing                                141.0  176.754269   70.408843   
Solutionised                                   4.0  228.750000    8.539126   
Solutionised  + Artificially peak aged       352.0  408.878138  144.242764   
Solutionised + Artificially over aged        208.0  466.242529  114.138564   
Solutionised + Cold Worked + Naturally aged   57.0  426.206062   91.479955   
Solutionised + Naturally aged                 86.0  320.173987  111.112826   
Strain Harderned (Hard)                      197.0  249.493873   79.995948   
Strain hardened     

In [14]:
import pandas as pd
import numpy as np

# Define processing condition mapping
processing_conditions = {
    "No Processing":                          (None, None, None, None, 0),
    "Strain hardened (Hard)":                 (None, None, None, None, 1.0),
    "Strain hardened":                        (None, None, None, None, 0.5),
    "Solutionised":                           (540, 1.0, None, None, 0),
    "Naturally aged":                         (None, None, 25, 168, 0),
    "Artificial aged":                        (None, None, 160, 8, 0),
    "Solutionised + Naturally aged":          (540, 1.0, 25, 168, 0),
    "Solutionised + Artificially peak aged":  (540, 1.0, 160, 8, 0),
    "Solutionised + Artificially over aged":  (540, 1.0, 200, 24, 0),
    "Solutionised + Cold Worked + Naturally aged": (540, 1.0, 25, 168, 0.3),
}

# Function to apply processing parameters
def add_processing_conditions(df, processing_column="Processing"):
    # Unpack each condition into separate columns
    df["solution_temp"] = df[processing_column].map(lambda x: processing_conditions.get(x, (None, None, None, None, None))[0])
    df["solution_time"] = df[processing_column].map(lambda x: processing_conditions.get(x, (None, None, None, None, None))[1])
    df["aging_temp"] = df[processing_column].map(lambda x: processing_conditions.get(x, (None, None, None, None, None))[2])
    df["aging_time"] = df[processing_column].map(lambda x: processing_conditions.get(x, (None, None, None, None, None))[3])
    df["strain_hardening_index"] = df[processing_column].map(lambda x: processing_conditions.get(x, (None, None, None, None, None))[4])
    return df

# Example usage:
df = pd.read_csv("al_data.csv")
df = add_processing_conditions(df)
df.head()


,Unnamed: 0,Processing,Ag,Al,B,Be,Bi,Cd,Co,Cr,...,Zr,Elongation (%),Tensile Strength (MPa),Yield Strength (MPa),class,solution_temp,solution_time,aging_temp,aging_time,strain_hardening_index
0,0,Solutionised + Artificially peak aged,0.0,0.88011,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0012,16.8,651.6,583.3,2,NaN,NaN,NaN,NaN,NaN
1,1,Solutionised + Artificially over aged,0.0,0.88011,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0012,15.4,557.0,513.0,4,540.0,1.0,200.0,24.0,0.0
2,2,Solutionised + Artificially peak aged,0.0,0.99450,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0000,10.5,320.0,300.0,2,NaN,NaN,NaN,NaN,NaN
3,3,Solutionised + Artificially peak aged,0.0,0.99250,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0000,4.5,280.0,265.0,2,NaN,NaN,NaN,NaN,NaN
4,4,Solutionised + Artificially peak aged,0.0,0.99000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0000,7.0,325.0,290.0,2,NaN,NaN,NaN,NaN,NaN


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1154 entries, 0 to 1153
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              1154 non-null   int64  
 1   Processing              1154 non-null   object 
 2   Ag                      1154 non-null   float64
 3   Al                      1154 non-null   float64
 4   B                       1154 non-null   float64
 5   Be                      1154 non-null   float64
 6   Bi                      1154 non-null   float64
 7   Cd                      1154 non-null   float64
 8   Co                      1154 non-null   float64
 9   Cr                      1154 non-null   float64
 10  Cu                      1154 non-null   float64
 11  Er                      1154 non-null   float64
 12  Eu                      1154 non-null   float64
 13  Fe                      1154 non-null   float64
 14  Ga                      1154 non-null   

In [16]:
# Drop the 'Processing' column from df_scaled if it exists
if 'Processing' in df_scaled.columns:
    df_scaled = df_scaled.drop(columns=['Processing'])

In [21]:
df_final = df.drop(columns=['Processing']) if 'Processing' in df.columns else df
df_final.to_csv('final_aluminum_data.csv', index=False)

In [23]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1154 entries, 0 to 1153
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              1154 non-null   int64  
 1   Ag                      1154 non-null   float64
 2   Al                      1154 non-null   float64
 3   B                       1154 non-null   float64
 4   Be                      1154 non-null   float64
 5   Bi                      1154 non-null   float64
 6   Cd                      1154 non-null   float64
 7   Co                      1154 non-null   float64
 8   Cr                      1154 non-null   float64
 9   Cu                      1154 non-null   float64
 10  Er                      1154 non-null   float64
 11  Eu                      1154 non-null   float64
 12  Fe                      1154 non-null   float64
 13  Ga                      1154 non-null   float64
 14  Li                      1154 non-null   